# Pipeline scratchpad

The same code the scheduler runs, importable from here. The Pipeline Airflow
add-on refreshes `/share/pipeline-airflow/lib` on every start, so this notebook
and the DAG can never disagree about what the merge does.

**Before running**, add these to the JupyterLab add-on's `init_commands` so they
survive a restart — no JVM is needed, the Spark Connect client is pure Python:

```yaml
init_commands:
  - pip install --no-cache-dir "pyspark-client==4.1.3"
  - pip install --no-cache-dir --no-deps "delta-spark==4.3.1"
```


In [ ]:
import sys
sys.path.insert(0, "/share/pipeline-airflow/lib")

from trackers_feed import discover_base_url, get, read_batch
from trackers_merge import merge_batch


## 1. Read a tracker

`discover_base_url` works out the address from the *Airflow* add-on's hostname,
so it returns `None` here — this container is named differently. Name the tracker
explicitly instead. The token is the `api_token` from that add-on's own config.


In [ ]:
TOKEN = ""  # paste the tracker's api_token
BASE = "http://6753e04e-gym-tracker:8099"  # <prefix>-gym-tracker, no host port needed

export = get("gym_tracker", BASE, TOKEN, "/api/export")
print(export["max_seq"], "seq;", len(export["tables"]), "tables")
sorted(export["tables"])


## 2. Query the Delta tables

The lakehouse the hourly DAG writes. `data` is kept as a JSON *string* on purpose:
both apps gain columns regularly, and inferring a schema per batch would eventually
produce two batches that disagree about a type.


In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.remote("sc://172.30.32.1:15002").getOrCreate()
logs = spark.read.format("delta").load("s3a://lakehouse/gym_tracker/workout_logs")
logs.count()


### Parsing `data`

Give it an explicit schema for the columns you actually want. Anything absent
comes back null rather than failing, so this survives the apps adding columns.


In [ ]:
SCHEMA = "id long, ts string, exercise_id long, sets long, reps long, duration_sec long, hr_avg long, hr_max long"

workouts = (
    logs.where("deleted_at IS NULL")
        .select(F.from_json("data", SCHEMA).alias("w"))
        .select("w.*")
)
workouts.orderBy(F.col("ts").desc()).show(10, truncate=False)


### Reps per day, as a starting point


In [ ]:
(workouts
 .withColumn("day", F.substring("ts", 1, 10))
 .groupBy("day")
 .agg(F.sum(F.col("sets") * F.col("reps")).alias("reps"),
      F.round(F.avg("hr_avg"), 1).alias("hr_avg"))
 .orderBy("day")
 .show(30, truncate=False))


## 3. Try a merge without touching the real tables

`merge_batch` takes the lakehouse root as an argument, so point it somewhere
scratch. This is how to test a change to `trackers_merge.py` against real data
before letting the DAG near it.


In [ ]:
# merge_batch(spark, "gym_tracker",
#             "s3a://raw/gym_tracker/<a timestamped prefix>",
#             "s3a://lakehouse/_scratch")
